In [ ]:
import sys
from pathlib import Path

# Assume the kernel cwd is this notebook's folder (neighbor of setup_fiftyone_dev_env.py).
sys.path.insert(0, str(Path.cwd()))

from setup_fiftyone_dev_env import setup

setup()

In [ ]:
import time

import numpy as np
import plotly.graph_objects as go

import fiftyone as fo

TS_NAME = "sdk_notebook_sine_demo"
# Window loaded from MongoDB (after save) for timing demo
QUERY_START_S = 0.5
QUERY_END_S = 2.5

# Synthetic sine: timestamps in seconds, one channel
DURATION_S = 60.0
N_SAMPLES = 1000000
FREQ_HZ = 0.5

In [ ]:
if fo.TimeSeries.exists(TS_NAME):
    fo.TimeSeries.delete(TS_NAME)

In [ ]:
t = np.linspace(0.0, DURATION_S, N_SAMPLES)
y = np.sin(2 * np.pi * FREQ_HZ * t)

# In-memory only — no DB until save()
ts = fo.TimeSeries.from_channels(
    TS_NAME,
    {"sine": fo.TimeSeriesChannel("sine", t, y)},
)

t0 = time.perf_counter()
ts.save()
save_s = time.perf_counter() - t0
print(f"save() → MongoDB: {save_s * 1000:.1f} ms ({len(t)} points)")

In [ ]:

timestamps, values, channel_names = ts.to_numpy()

values_mean = np.mean(values, axis=0)
values_std = np.std(values, axis=0)

print(f"  Time span: {timestamps[0]:.1f}s → {timestamps[-1]:.1f}s")
for name, m, s in zip(channel_names, values_mean, values_std):
    print(f"  {name}: mean={m:.6g}, std={s:.6g}")

In [ ]:
# Drop in-memory copy — load window from DB only (separate timed operation)
del ts

In [ ]:
t0 = time.perf_counter()
window = fo.TimeSeries.load(
    TS_NAME,
    start=QUERY_START_S,
    end=QUERY_END_S,
)
load_s = time.perf_counter() - t0
ch0 = window.channel_names[0]
n = len(window[ch0])
print(
    f"fo.TimeSeries.load({QUERY_START_S!r}, {QUERY_END_S!r}) from MongoDB: "
    f"{load_s * 1000:.1f} ms → {n} samples on channel {ch0!r}"
)



In [ ]:
fig = go.Figure(window.to_plotly())
fig.update_layout(
    title=f"{TS_NAME} [{QUERY_START_S}s–{QUERY_END_S}s] from DB — sine SDK demo"
)
fig.show()